# W3D5: Hyperparameter Tuning — GridSearch & RandomSearch

## Objective

This project demonstrates hyperparameter tuning using GridSearchCV
and RandomizedSearchCV.

The performance of Support Vector Machine (SVM) and
K-Nearest Neighbours (KNN) classifiers is compared using
cross-validation and test-set evaluation.

In [1]:
# Import required libraries

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the Breast Cancer dataset from scikit-learn

data = load_breast_cancer()

# Create feature DataFrame
X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

# Create target Series
y = pd.Series(
    data.target,
    name="target"
)

print("Dataset loaded successfully!")
print("Dataset shape:", X.shape)
print("Number of classes:", y.nunique())
print("\nTarget classes:", data.target_names)

Dataset loaded successfully!
Dataset shape: (569, 30)
Number of classes: 2

Target classes: ['malignant' 'benign']


In [3]:
# Basic dataset exploration

print("First 5 rows:")
display(X.head())

print("\nDataset information:")
print(X.info())

print("\nMissing values:")
print(X.isnull().sum().sum())

print("\nClass distribution:")
print(y.value_counts())

First 5 rows:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness erro

In [4]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [5]:
# Create baseline SVM pipeline
# StandardScaler prevents features with larger values
# from dominating the model.

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])

# Train SVM
svm_pipeline.fit(X_train, y_train)

# Make predictions
svm_predictions = svm_pipeline.predict(X_test)

# Calculate accuracy
svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)


# Create baseline KNN pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

# Train KNN
knn_pipeline.fit(X_train, y_train)

# Make predictions
knn_predictions = knn_pipeline.predict(X_test)

# Calculate accuracy
knn_accuracy = accuracy_score(
    y_test,
    knn_predictions
)


print("Baseline Model Results")
print("-" * 40)
print(f"SVM Accuracy : {svm_accuracy:.4f}")
print(f"KNN Accuracy : {knn_accuracy:.4f}")

Baseline Model Results
----------------------------------------
SVM Accuracy : 0.9825
KNN Accuracy : 0.9561


In [6]:
# Hyperparameter tuning for SVM using GridSearchCV

svm_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train GridSearchCV
svm_grid.fit(X_train, y_train)

print("SVM GridSearchCV completed!")
print("\nBest Parameters:")
print(svm_grid.best_params_)

print(
    f"\nBest Cross-Validation Accuracy: "
    f"{svm_grid.best_score_:.4f}"
)

SVM GridSearchCV completed!

Best Parameters:
{'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}

Best Cross-Validation Accuracy: 0.9780


In [7]:
# Hyperparameter tuning for SVM using RandomizedSearchCV

svm_random_params = {
    "model__C": np.logspace(-2, 2, 20),
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"]
}

svm_random = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_random_params,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

# Train RandomizedSearchCV
svm_random.fit(X_train, y_train)

print("SVM RandomizedSearchCV completed!")

print("\nBest Parameters:")
print(svm_random.best_params_)

print(
    f"\nBest Cross-Validation Accuracy: "
    f"{svm_random.best_score_:.4f}"
)

SVM RandomizedSearchCV completed!

Best Parameters:
{'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__C': np.float64(3.359818286283781)}

Best Cross-Validation Accuracy: 0.9736


In [8]:
# Hyperparameter tuning for KNN using GridSearchCV

knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train GridSearchCV
knn_grid.fit(X_train, y_train)

print("KNN GridSearchCV completed!")

print("\nBest Parameters:")
print(knn_grid.best_params_)

print(
    f"\nBest Cross-Validation Accuracy: "
    f"{knn_grid.best_score_:.4f}"
)

KNN GridSearchCV completed!

Best Parameters:
{'model__metric': 'euclidean', 'model__n_neighbors': 7, 'model__weights': 'uniform'}

Best Cross-Validation Accuracy: 0.9714


In [9]:
# Hyperparameter tuning for KNN using RandomizedSearchCV

knn_random_params = {
    "model__n_neighbors": list(range(1, 31)),
    "model__weights": ["uniform", "distance"],
    "model__metric": [
        "euclidean",
        "manhattan",
        "minkowski"
    ]
}

knn_random = RandomizedSearchCV(
    estimator=knn_pipeline,
    param_distributions=knn_random_params,
    n_iter=15,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

# Train RandomizedSearchCV
knn_random.fit(X_train, y_train)

print("KNN RandomizedSearchCV completed!")

print("\nBest Parameters:")
print(knn_random.best_params_)

print(
    f"\nBest Cross-Validation Accuracy: "
    f"{knn_random.best_score_:.4f}"
)

KNN RandomizedSearchCV completed!

Best Parameters:
{'model__weights': 'distance', 'model__n_neighbors': 8, 'model__metric': 'euclidean'}

Best Cross-Validation Accuracy: 0.9736


In [10]:
# Evaluate the best estimators obtained from both search methods

models = {
    "SVM GridSearch": svm_grid.best_estimator_,
    "SVM RandomSearch": svm_random.best_estimator_,
    "KNN GridSearch": knn_grid.best_estimator_,
    "KNN RandomSearch": knn_random.best_estimator_
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "Test Accuracy": accuracy
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

print("Model Comparison")
print("=" * 50)

display(results_df)

Model Comparison


,Model,Test Accuracy
0,SVM GridSearch,0.982456
1,SVM RandomSearch,0.982456
2,KNN GridSearch,0.973684
3,KNN RandomSearch,0.973684


In [11]:
# Compare baseline models with the tuned models

baseline_results = pd.DataFrame({
    "Model": [
        "Baseline SVM",
        "Baseline KNN"
    ],
    "Test Accuracy": [
        svm_accuracy,
        knn_accuracy
    ]
})

comparison_df = pd.concat(
    [baseline_results, results_df],
    ignore_index=True
)

comparison_df = comparison_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

print("Baseline vs Tuned Model Comparison")
print("=" * 50)

display(comparison_df)

Baseline vs Tuned Model Comparison


,Model,Test Accuracy
0,Baseline SVM,0.982456
1,SVM GridSearch,0.982456
2,SVM RandomSearch,0.982456
3,KNN GridSearch,0.973684
4,KNN RandomSearch,0.973684
5,Baseline KNN,0.956140


In [12]:
# Select the model with the highest test accuracy

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_predictions = best_model.predict(X_test)

best_accuracy = accuracy_score(
    y_test,
    best_predictions
)

print("Best Model")
print("=" * 50)
print("Model:", best_model_name)
print(f"Test Accuracy: {best_accuracy:.4f}")

Best Model
Model: SVM GridSearch
Test Accuracy: 0.9825


In [13]:
# Detailed evaluation of the best model

print("Classification Report")
print("=" * 50)

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=data.target_names
    )
)

Classification Report
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [14]:
# Display the confusion matrix for the best model

cm = confusion_matrix(
    y_test,
    best_predictions
)

print("Confusion Matrix")
print("=" * 50)

display(
    pd.DataFrame(
        cm,
        index=[
            "Actual Malignant",
            "Actual Benign"
        ],
        columns=[
            "Predicted Malignant",
            "Predicted Benign"
        ]
    )
)

Confusion Matrix


,Predicted Malignant,Predicted Benign
Actual Malignant,41,1
Actual Benign,1,71


## Conclusion

Hyperparameter tuning was performed using GridSearchCV and
RandomizedSearchCV for both Support Vector Machine (SVM) and
K-Nearest Neighbours (KNN).

The models were evaluated using 5-fold cross-validation and
test-set accuracy.

GridSearchCV performs an exhaustive search over the specified
parameter combinations, while RandomizedSearchCV evaluates a
random selection of parameter combinations.

The tuned models were compared with the baseline SVM and KNN
models to determine whether hyperparameter optimization improved
performance.

The model with the highest test accuracy was selected as the
final model.